# E01 Matrix Sensing Benchmark - PyTorch

Notebook-first Matrix Sensing experiment. The autograd problem lives in `problems/`; this notebook owns the experiment grid, optimizer step, run loop, joblib parallelism, plots, and conclusion.

## Imports

In [ ]:
import os
import pathlib
import sys
import time
for name in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(name, "1")

import joblib
import IPython.display
import matplotlib.pyplot as plt
import pandas as pd
import torch

PROJECT = pathlib.Path.cwd().resolve()
if not (PROJECT / "problems").exists():
    PROJECT = PROJECT.parent.resolve()
sys.path.insert(0, str(PROJECT))

import optimizers
import plotting
import problems.MatrixConstruction
import problems.MatrixSensing
import util

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

print(f"project = {PROJECT}")
print(f"torch   = {torch.__version__}")
print(f"joblib  = {joblib.__version__}")

## Parameters And Runs

Before execution, `runs` is one row per independent run. The execution cell replaces it with a long per-step table: one row per `(run_id, step)`.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE_NAME = "float64"
ALGOS = ["Muon", "Muon-Exact", "Shampoo", "Adam", "SGD"]
DIMS = [50, 60, 70]
SEEDS = list(range(10))
SMOKE_TEST = False
SMOKE_TEST_MAX_STEPS = 10
FULL_ITERS = 2000

BASE_SPEC = dict(
    rank=5,
    lr=0.01,
    noise=0.0,
    dist="normal",
    spectrum="hard-cutoff",
    kappa=1.0,
    init_scale=0.01,
    iters=SMOKE_TEST_MAX_STEPS if SMOKE_TEST else FULL_ITERS,
    early_stop=True,
    early_stop_min_steps=20,
    early_stop_patience=20,
    early_stop_min_delta=1e-3,
    device_type=DEVICE.type,
    dtype_name=DTYPE_NAME,
)
NUM_WORKERS = min(8, os.cpu_count() or 1)
JOBLIB_BACKEND = "loky"

runs = pd.DataFrame([
    {**BASE_SPEC, "algo": algo, "d": d, "seed": seed}
    for algo in ALGOS
    for d in DIMS
    for seed in SEEDS
])
runs["problem"] = "MatrixSensing"
runs["r"] = runs["rank"]
runs["m_meas"] = 2 * runs["d"] * runs["rank"]
runs.insert(0, "run_id", range(len(runs)))
runs = runs[[
    "run_id", "problem", "algo", "d", "rank", "r", "m_meas", "seed", "lr", "iters",
    "early_stop", "early_stop_min_steps", "early_stop_patience", "early_stop_min_delta",
    "noise", "dist", "spectrum", "kappa", "init_scale", "device_type", "dtype_name",
]]

print(f"device={DEVICE}, workers={NUM_WORKERS}, backend={JOBLIB_BACKEND}")
print(f"smoke_test={SMOKE_TEST}, smoke_test_max_steps={SMOKE_TEST_MAX_STEPS}")
print(f"runs={len(runs)}, max_iters={BASE_SPEC['iters']}, max_total_steps={len(runs) * BASE_SPEC['iters']}")
print(
    "early_stop=patience on absolute loss improvement "
    f"(min_steps={BASE_SPEC['early_stop_min_steps']}, "
    f"patience={BASE_SPEC['early_stop_patience']}, "
    f"min_delta={BASE_SPEC['early_stop_min_delta']})"
)
IPython.display.display(runs)

## Problem

Target matrix:

$$
X^\star = U \operatorname{diag}(s)V^\top
$$

Measurements:

$$
y_i = \langle A_i, X^\star \rangle + \varepsilon_i
$$

Loss:

$$
f(X) = \frac{1}{2m}\sum_{i=1}^{m}(\langle A_i, X \rangle-y_i)^2
$$

## Single Run

In [ ]:
def single_run(run):
    run = dict(run)
    d, rank, seed, iters = int(run["d"]), int(run["rank"]), int(run["seed"]), int(run["iters"])
    early_stop = bool(run["early_stop"])
    early_stop_min_steps = int(run["early_stop_min_steps"])
    early_stop_patience = int(run["early_stop_patience"])
    early_stop_min_delta = float(run["early_stop_min_delta"])
    device = torch.device(run["device_type"])
    dtype = dtype_from_name(run["dtype_name"])
    configure_torch(dtype)

    def initialize_problem(run):
        return problems.MatrixSensing.make_matrix_sensing_problem(
            d,
            rank,
            noise=float(run["noise"]),
            dist=run["dist"],
            spectrum=run["spectrum"],
            kappa=float(run["kappa"]),
            seed=seed,
            device=device,
            dtype=dtype,
        )

    def optimization(problem, run):
        x = torch.nn.Parameter(problems.MatrixConstruction.randn((d, d), seed + 3000, device, dtype) * float(run["init_scale"]))
        state = {
            "problem": problem,
            "x": x,
            "optimizer": make_optimizer(run["algo"], [x], float(run["lr"]), rank),
            "step": 0,
            "best_loss": None,
            "early_stop_wait": 0,
        }

        def step(state):
            state["optimizer"].zero_grad(set_to_none=True)
            loss = state["problem"].loss(state["x"])
            loss.backward()
            state["grad_norm"] = float(state["x"].grad.detach().norm().cpu())
            state["optimizer"].step()
            state["step"] += 1
            state["loss"] = float(loss.detach().cpu())
            return state

        def update_early_stop_state(state):
            if state["best_loss"] is None:
                state["best_loss"] = state["loss"]
                return state
            absolute_improvement = state["best_loss"] - state["loss"]
            if absolute_improvement >= early_stop_min_delta:
                state["best_loss"] = state["loss"]
                state["early_stop_wait"] = 0
            else:
                state["early_stop_wait"] += 1
            return state

        def should_stop(state):
            return (
                early_stop
                and state["step"] >= early_stop_min_steps
                and state["early_stop_wait"] >= early_stop_patience
            )

        def stepwise_data(state, stop_reason=""):
            return {
                **run,
                "step": state["step"],
                "loss": state["loss"],
                "grad_norm": state["grad_norm"],
                "best_loss": state["best_loss"],
                "early_stop_wait": state["early_stop_wait"],
                "elapsed_s": time.time() - t0,
                "stop_reason": stop_reason,
            }

        records = []
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        t0 = time.time()
        for _ in range(iters):
            state = step(state)
            state = update_early_stop_state(state)
            stop_reason = "early_stop_patience" if should_stop(state) else ""
            records.append(stepwise_data(state, stop_reason))
            if stop_reason:
                break
        if records and not records[-1]["stop_reason"]:
            records[-1]["stop_reason"] = "max_iters"
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        return pd.DataFrame(records)

    problem = initialize_problem(run)
    run.update(d=d, rank=rank, r=rank, m_meas=problem.m_meas, seed=seed, iters=iters)
    return optimization(problem, run)

## Runtime Helpers

In [ ]:
def dtype_from_name(name):
    dtype = getattr(torch, name)
    if not isinstance(dtype, torch.dtype):
        raise ValueError(f"unknown torch dtype: {name}")
    return dtype


def configure_torch(dtype):
    torch.set_default_dtype(dtype)
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass


def make_optimizer(algo, params, lr, rank):
    if algo == "Muon" and hasattr(torch.optim, "Muon"):
        return torch.optim.Muon(params, lr=lr, weight_decay=0.0, momentum=0.9, nesterov=False, ns_steps=5)
    if algo == "Muon":
        return optimizers.MuonExact(params, lr=lr, momentum=0.9, variant="newton_schulz", ns_steps=5)
    if algo in {"Muon-Exact", "MuonExact"}:
        return optimizers.MuonExact(params, lr=lr, momentum=0.9, variant="exact")
    if algo == "Shampoo":
        return optimizers.Shampoo(params, lr=lr, beta2=0.9, epsilon=1e-8)
    if algo == "Adam":
        return torch.optim.Adam(params, lr=lr)
    if algo == "SGD":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    raise ValueError(f"unknown algo: {algo}")

## Plotting API

In [ ]:
for module_name in list(sys.modules):
    if module_name == "plotting" or module_name.startswith("plotting."):
        sys.modules.pop(module_name)

import plotting


def show_figure(fig):
    IPython.display.display(fig)
    plt.close(fig)

## Execute

This cell runs every row in `runs` through `single_run`. After it runs, `runs` becomes the long per-step result table.

In [ ]:
runs = util.run_experiments(runs, single_run, num_workers=NUM_WORKERS, backend=JOBLIB_BACKEND, algo_order=ALGOS)
IPython.display.display(runs)

## Result Tables

In [ ]:
ordered_runs = runs.sort_values(["run_id", "step"])

run_summary = ordered_runs.groupby("run_id", as_index=False).first()
run_summary = run_summary.drop(
    columns=["step", "loss", "grad_norm", "best_loss", "early_stop_wait", "elapsed_s", "stop_reason"]
)
run_summary = run_summary.merge(
    ordered_runs.groupby("run_id", as_index=False).agg(
        final_loss=("loss", "last"),
        min_loss=("loss", "min"),
        actual_steps=("step", "last"),
        time_s=("elapsed_s", "last"),
        stop_reason=("stop_reason", "last"),
    ),
    on="run_id",
)
run_summary["stopped_early"] = run_summary["actual_steps"] < run_summary["iters"]
run_summary["algo"] = pd.Categorical(run_summary["algo"], categories=ALGOS, ordered=True)
run_summary = run_summary.sort_values(["algo", "d", "seed"]).reset_index(drop=True)
run_summary["algo"] = run_summary["algo"].astype(str)

trajectories = {
    (str(algo), int(d), int(seed)): {
        "loss": group["loss"].tolist(),
        "grad_norm": group["grad_norm"].tolist(),
    }
    for (algo, d, seed), group in ordered_runs.groupby(["algo", "d", "seed"], sort=False)
}
summary = plotting.summary_table(run_summary)

IPython.display.display(runs)
IPython.display.display(run_summary)
IPython.display.display(summary)

## Color Key

In [ ]:
fig, axes = plotting.plot_color_key(run_summary)
show_figure(fig)

## Metric Overview

In [ ]:
fig, axes = plotting.plot_metric_overview(run_summary, loss_log_y=False)
show_figure(fig)

fig, axes = plotting.plot_metric_overview(run_summary, loss_log_y=True)
show_figure(fig)

## Actual Steps Bar

In [ ]:
fig, ax = plotting.plot_metric_bar(run_summary, "actual_steps_mean", "Mean executed steps by algorithm and dimension", "steps")
show_figure(fig)

## Wall Clock Bar

In [ ]:
fig, ax = plotting.plot_metric_bar(run_summary, "time_s_mean", "Mean wall-clock by algorithm and dimension", "seconds")
show_figure(fig)

## Minimum Loss Bar

In [ ]:
fig, ax = plotting.plot_metric_bar(run_summary, "min_loss_mean", "Mean minimum loss by algorithm and dimension (linear scale)", "loss")
show_figure(fig)

fig, ax = plotting.plot_metric_bar(run_summary, "min_loss_mean", "Mean minimum loss by algorithm and dimension (log scale)", "loss", log_y=True)
show_figure(fig)

## Final Loss Bar

In [ ]:
fig, ax = plotting.plot_metric_bar(run_summary, "final_loss_mean", "Mean final loss by algorithm and dimension (linear scale)", "loss")
show_figure(fig)

fig, ax = plotting.plot_metric_bar(run_summary, "final_loss_mean", "Mean final loss by algorithm and dimension (log scale)", "loss", log_y=True)
show_figure(fig)

## Same Dimension Algorithm Comparisons

In [ ]:
for d in plotting.ordered_dims_in(run_summary):
    for log_y in (False, True):
        fig, ax = plotting.plot_algorithms_for_dimension(trajectories, d, log_y=log_y)
        show_figure(fig)

## Same Algorithm Dimension Comparisons

In [ ]:
for algo in plotting.ordered_algos_in(run_summary):
    for log_y in (False, True):
        fig, ax = plotting.plot_dimensions_for_algorithm(trajectories, algo, log_y=log_y)
        show_figure(fig)

## All Mean Curves

In [ ]:
fig, ax = plotting.plot_all_mean_curves_combined(trajectories, log_y=False)
show_figure(fig)

fig, ax = plotting.plot_all_mean_curves_combined(trajectories, log_y=True)
show_figure(fig)

## Algorithm-Dimension Grid

In [ ]:
fig, axes = plotting.plot_algorithm_dimension_grid(trajectories, log_y=False)
show_figure(fig)

fig, axes = plotting.plot_algorithm_dimension_grid(trajectories, log_y=True)
show_figure(fig)

## Seed Variability By Dimension

In [ ]:
for d in plotting.ordered_dims_in(run_summary):
    for log_y in (False, True):
        fig, ax = plotting.plot_seed_variability_for_dimension(trajectories, d, log_y=log_y)
        show_figure(fig)

## Conclusion

In [ ]:
def conclusion_markdown(run_summary):
    summary = plotting.summary_table(run_summary)
    max_steps = int(run_summary["iters"].sum())
    actual_steps = int(run_summary["actual_steps"].sum())
    stopped = int(run_summary["stopped_early"].sum())
    lines = [
        "### Result Summary",
        "",
        f"- Runs: `{len(run_summary)}`",
        f"- Methods: `{', '.join(sorted(run_summary['algo'].unique()))}`",
        f"- Max iterations per run: `{int(run_summary['iters'].iloc[0])}`",
        f"- Max optimizer-step budget: `{max_steps}`",
        f"- Executed optimizer steps: `{actual_steps}` (`{max_steps - actual_steps}` skipped by early stopping)",
        f"- Early-stopped runs: `{stopped}/{len(run_summary)}`",
        "",
    ]
    for d in sorted(run_summary["d"].unique()):
        sub = summary[summary["d"] == d]
        fewest_steps = sub.loc[sub["actual_steps_mean"].idxmin()]
        best_time = sub.loc[sub["time_s_mean"].idxmin()]
        best_loss = sub.loc[sub["min_loss_mean"].idxmin()]
        lines.append(
            f"- d={d}: fewest executed steps is `{fewest_steps.algo}` at `{fewest_steps.actual_steps_mean:.1f}` steps; "
            f"fastest wall-clock is `{best_time.algo}` at `{best_time.time_s_mean:.3f}s`; "
            f"lowest min_loss is `{best_loss.algo}` at `{best_loss.min_loss_mean:.3e}`."
        )
    lines.append("")
    lines.append("Muon vs Muon-Exact checks whether the Newton-Schulz approximation tracks the exact SVD polar direction.")
    lines.append("Compare actual steps, wall-clock, and minimum loss together when judging optimizer efficiency.")
    return "\n".join(lines)


IPython.display.display(IPython.display.Markdown(conclusion_markdown(run_summary)))